# Fase 9 — Análisis en profundidad de Golazo (Shorts vs. vídeos largos)

Continúa el EDA de la Fase 8 separando siempre **Shorts** de **vídeos largos**: son formatos que compiten en feeds distintos, con dinámicas de duración/retención muy diferentes, y mezclarlos sin distinguir sesga cualquier comparación de rendimiento (ya lo vimos: los vídeos con duración muy cercana a 0 minutos son probablemente Shorts, no un error de los datos).

**Requisito previo:** base de datos `golazo_growup` ya cargada (`python -m src.cargar_datos`).

## 0. Advertencia sobre granularidad de los datos

- `evolucion_diaria`, `trafico`, `demografia`, `ingresos` → agregados a **nivel de canal**, por periodo completo (no por vídeo, no por categoría, no por formato).
- `retencion_audiencia` → ya cubre los **338 vídeos** completos (ampliado respecto a la versión anterior de este notebook, que solo tenía una muestra de 15).
- `video.views_totales` → acumulado de **toda la vida** del vídeo, no vistas del día concreto.
- **Shorts vs. largos:** YouTube considera Short a cualquier vídeo de **hasta 3 minutos (180s)** desde octubre de 2024. Se usa ese umbral para clasificar cada vídeo del catálogo. Es una aproximación razonable (no tenemos el indicador exacto de formato vertical/#Shorts que sí tiene la API real), pero con datos sintéticos es el proxy más fiable disponible.

## 0.1 Carga de datos y columnas derivadas

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

from src import db_connection as dbc

url = dbc.DATABASE_URL or (
    f"postgresql+psycopg2://{dbc.DB_CONFIG['user']}:{dbc.DB_CONFIG['password']}"
    f"@{dbc.DB_CONFIG['host']}:{dbc.DB_CONFIG['port']}/{dbc.DB_CONFIG['dbname']}"
)
if url.startswith('postgresql://'):
    url = url.replace('postgresql://', 'postgresql+psycopg2://', 1)
engine = create_engine(url)

canal = pd.read_sql('SELECT * FROM canal', engine)
video = pd.read_sql('SELECT * FROM video', engine, parse_dates=['fecha_publicacion'])
video_tag = pd.read_sql('SELECT * FROM video_tag', engine)
evolucion = pd.read_sql('SELECT * FROM evolucion_diaria', engine, parse_dates=['fecha'])
retencion = pd.read_sql('SELECT * FROM retencion_audiencia', engine)
trafico = pd.read_sql('SELECT * FROM trafico', engine)
ingresos = pd.read_sql('SELECT * FROM ingresos', engine, parse_dates=['fecha'])

# Columnas derivadas
evolucion['suscriptores_netos'] = evolucion['subscribers_gained'] - evolucion['subscribers_lost']
evolucion['dia_semana'] = evolucion['fecha'].dt.day_name()
evolucion['es_finde'] = evolucion['dia_semana'].isin(['Saturday', 'Sunday'])

UMBRAL_SHORT_SEGUNDOS = 180  # YouTube: máximo de un Short desde octubre de 2024
video['es_short'] = video['duracion_segundos'] <= UMBRAL_SHORT_SEGUNDOS
video['formato'] = video['es_short'].map({True: 'Short', False: 'Largo'})

video['ratio_likes_vista'] = video['likes'] / video['views_totales']
video['ratio_comentarios_vista'] = video['comentarios'] / video['views_totales']

# Z-score de vistas anómalas calculado por (categoría + formato), no solo por categoría:
# mezclar Shorts y largos en el mismo grupo de referencia sesgaría la detección de anomalías,
# porque el formato por sí solo ya explica gran parte de la diferencia de vistas/duración.
video['z_views_categoria_formato'] = video.groupby(['categoria', 'formato'])['views_totales'].transform(
    lambda s: (s - s.mean()) / s.std() if s.std() > 0 else 0
)

video['dia_semana_publicacion'] = video['fecha_publicacion'].dt.day_name()
fecha_corte = video['fecha_publicacion'].max() - pd.Timedelta(days=30)
video['es_reciente'] = video['fecha_publicacion'] > fecha_corte

retencion_media_por_video = retencion.groupby('video_id')['audience_watch_ratio'].mean().rename('retencion_media')
video = video.merge(retencion_media_por_video, on='video_id', how='left')

print(f'Vídeos totales: {len(video)}')
print(video['formato'].value_counts())

## 0.2 Comparativa general: Shorts vs. vídeos largos

Panorama de partida antes de entrar en cada punto — así cada hallazgo posterior se lee ya sabiendo si el formato en sí mismo explica una diferencia.

In [ ]:
comparativa_formato = video.groupby('formato')[
    ['views_totales', 'likes', 'comentarios', 'ratio_likes_vista',
     'ratio_comentarios_vista', 'duracion_segundos', 'retencion_media']
].mean()
comparativa_formato['n_videos'] = video['formato'].value_counts()
print('Shorts vs. vídeos largos — medias generales:')
display(comparativa_formato.round(3))

In [ ]:
plt.figure(figsize=(8, 4))
video.groupby('formato')['views_totales'].mean().plot(kind='bar', color=['#B23A2E', '#2E6E9E'])
plt.ylabel('Vistas medias por vídeo')
plt.title('Vistas medias: Shorts vs. vídeos largos')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 1. Los dos días de mayor audiencia media: qué los caracteriza (Shorts vs. largos)

In [ ]:
top2_dias_audiencia = evolucion.sort_values('views', ascending=False).head(2)
print('Los 2 días de mayor audiencia media (por vistas):')
display(top2_dias_audiencia[['fecha', 'dia_semana', 'views', 'subscribers_gained', 'subscribers_lost', 'suscriptores_netos']])

media_views = evolucion['views'].mean()
media_subs = evolucion['subscribers_gained'].mean()
for _, fila in top2_dias_audiencia.iterrows():
    pct_views = (fila['views'] / media_views - 1) * 100
    pct_subs = (fila['subscribers_gained'] / media_subs - 1) * 100
    print(f"{fila['fecha'].date()} ({fila['dia_semana']}): {pct_views:+.0f}% vistas y {pct_subs:+.0f}% "
          f"suscriptores ganados respecto a la media del periodo.")

In [ ]:
videos_dias_top = video[video['fecha_publicacion'].isin(top2_dias_audiencia['fecha'])]
print(f'Vídeos publicados justo esos 2 días: {len(videos_dias_top)}')

if len(videos_dias_top):
    display(videos_dias_top[['titulo', 'categoria', 'formato', 'fecha_publicacion', 'views_totales', 'likes', 'comentarios']])
    print('\nReparto por formato ese día:')
    display(videos_dias_top['formato'].value_counts())
else:
    print('Ningún vídeo se publicó exactamente esos días — el pico de audiencia probablemente proviene de '
          'vídeos (Shorts o largos) publicados antes que siguen acumulando vistas, no de una subida nueva.')

**Para reforzar este punto:** con la granularidad actual no se puede aislar qué vídeo concreto generó las vistas de ese día. El dato real que haría falta es la Analytics API a nivel de vídeo con dimensión `day`. Aun así, saber si el contenido de esos días fue mayoritariamente Short o largo sí es información accionable: si los días fuertes coinciden con Shorts, el driver del pico es alcance rápido; si coinciden con vídeos largos, es más probable que sea un análisis o comentario específico el que enganchó a la audiencia.

## 2. Categoría 'Afición' en profundidad — separando Shorts de vídeos largos

In [ ]:
aficion = video[video['categoria'] == 'Afición']
print(f"Vídeos de la categoría Afición: {len(aficion)} de {len(video)} totales")
display(aficion['formato'].value_counts())

In [ ]:
comparativa_aficion_formato = aficion.groupby('formato')[
    ['views_totales', 'likes', 'comentarios', 'ratio_likes_vista',
     'ratio_comentarios_vista', 'duracion_segundos', 'retencion_media']
].mean()
print('Afición: Shorts vs. largos (medias dentro de la propia categoría):')
display(comparativa_aficion_formato.round(3))

In [ ]:
# Afición vs. resto del canal, PERO comparando cada formato con su propio equivalente
# (Shorts de Afición vs. Shorts del resto; largos de Afición vs. largos del resto)
for formato in ['Short', 'Largo']:
    grupo_aficion = aficion[aficion['formato'] == formato]
    grupo_resto = video[(video['categoria'] != 'Afición') & (video['formato'] == formato)]
    if len(grupo_aficion) == 0:
        print(f'Afición no tiene vídeos de tipo {formato} — no se puede comparar.')
        continue
    print(f"\n--- {formato}s: Afición (n={len(grupo_aficion)}) vs. resto del canal (n={len(grupo_resto)}) ---")
    print(f"  Vistas medias: {grupo_aficion['views_totales'].mean():.0f} vs. {grupo_resto['views_totales'].mean():.0f}")
    print(f"  Retención media: {grupo_aficion['retencion_media'].mean()*100:.1f}% vs. {grupo_resto['retencion_media'].mean()*100:.1f}%")
    print(f"  Likes/vista: {grupo_aficion['ratio_likes_vista'].mean()*100:.2f}% vs. {grupo_resto['ratio_likes_vista'].mean()*100:.2f}%")

In [ ]:
# Tags más frecuentes en Afición, separados por formato
for formato in ['Short', 'Largo']:
    ids_formato = aficion[aficion['formato'] == formato]['video_id']
    tags_formato = video_tag[video_tag['video_id'].isin(ids_formato)]['tag'].value_counts().head(3)
    print(f'Tags más usados en Afición ({formato}s):')
    display(tags_formato)

**Nota de granularidad:** se deja fuera la relación con ingresos y tráfico por categoría/formato porque en este proyecto ambos solo existen agregados a nivel de canal/día (Fase 3) — incluirlos aquí sería repartir una cifra de canal de forma arbitraria, no medirla.

## 3. ¿Los vídeos con vistas anormalmente altas coinciden con los días de mayor audiencia?

Se usa el z-score corregido por (categoría + formato) de la sección 0.1, para no confundir "es viral" con "es un Short" (los Shorts ya de por sí suelen tener una distribución de vistas distinta a los largos).

In [ ]:
picos_virales = video[video['z_views_categoria_formato'] > 2]
print(f'Vídeos con vistas anormalmente altas para su categoría Y formato: {len(picos_virales)} de {len(video)}')
display(picos_virales['formato'].value_counts())

coinciden = picos_virales[picos_virales['fecha_publicacion'].isin(top2_dias_audiencia['fecha'])]
print(f'\nDe esos, publicados justo en los 2 días de mayor audiencia: {len(coinciden)}')
if len(coinciden) == 0:
    print('No coinciden por fecha exacta — esperable, ya que views_totales acumula vistas de toda la vida '
          'del vídeo, no solo del día de publicación.')
else:
    display(coinciden[['titulo', 'categoria', 'formato', 'fecha_publicacion', 'views_totales']])

In [ ]:
# Perfil de los vídeos virales, comparado con el resto — SIEMPRE dentro del mismo formato,
# para que la diferencia observada sea sobre 'qué distingue a un viral', no sobre 'Shorts vs largos'.
for formato in ['Short', 'Largo']:
    virales_formato = video[(video['z_views_categoria_formato'] > 2) & (video['formato'] == formato)]
    resto_formato = video[(video['z_views_categoria_formato'] <= 2) & (video['formato'] == formato)]
    if len(virales_formato) == 0:
        print(f'No hay vídeos virales de tipo {formato} en este catálogo.')
        continue
    print(f"\n--- {formato}s virales (n={len(virales_formato)}) vs. resto de {formato.lower()}s (n={len(resto_formato)}) ---")
    print(f"  Duración media: {virales_formato['duracion_segundos'].mean():.0f}s vs. {resto_formato['duracion_segundos'].mean():.0f}s")
    print(f"  Likes/vista: {virales_formato['ratio_likes_vista'].mean()*100:.2f}% vs. {resto_formato['ratio_likes_vista'].mean()*100:.2f}%")
    print(f"  Retención media: {virales_formato['retencion_media'].mean()*100:.1f}% vs. {resto_formato['retencion_media'].mean()*100:.1f}%")
    print(f'  Categoría dominante entre los virales: {virales_formato["categoria"].mode().iloc[0]}')
    print(f'  Día de semana dominante entre los virales: {virales_formato["dia_semana_publicacion"].mode().iloc[0]}')

**Lectura profesional:** si dentro de su propio formato los virales también retienen mejor (no solo tienen más vistas), su éxito es de calidad de contenido y replicable. Si retienen igual o peor que el resto de su mismo formato pese a tener más vistas, el pico es más probablemente un golpe de alcance puntual, no una fórmula a repetir.

## 4. Top 5 de vídeos "perfectos" — un ranking por formato, no mezclados

Comparar un Short de 40 segundos con un análisis de 20 minutos en la misma escala normalizada penalizaría siempre a uno de los dos formatos de forma artificial. Se calcula el score **dentro de cada formato por separado**, así el Top 5 de cada uno compite solo contra vídeos de su misma naturaleza.

In [ ]:
def normalizar(serie):
    rango = serie.max() - serie.min()
    return (serie - serie.min()) / rango if rango > 0 else serie * 0

for formato in ['Short', 'Largo']:
    grupo = video[video['formato'] == formato].copy()
    grupo['score_perfecto'] = (
        normalizar(grupo['views_totales']) * 0.35
        + normalizar(grupo['likes']) * 0.25
        + normalizar(grupo['comentarios']) * 0.15
        + normalizar(grupo['retencion_media']) * 0.25
    )
    top5 = grupo.sort_values('score_perfecto', ascending=False).head(5)
    print(f'\nTOP 5 {formato}s "perfectos" (de {len(grupo)} {formato.lower()}s, score propio del formato):')
    display(top5[['titulo', 'categoria', 'views_totales', 'likes', 'comentarios',
                   'retencion_media', 'score_perfecto']].round(3))

**Por qué no incluye monetización por vídeo:** los ingresos (Fase 3) solo se definieron con dimensión `day` a nivel de canal, nunca `video`. Añadir una cifra de "euros por vídeo" aquí sería inventar un dato, no derivarlo. La YouTube Analytics API real sí permite pedir ingresos con dimensión `video` — sería el cambio a hacer en `analytics_client.py` si se necesita en una futura fase.

## 5. Qué se publica los días de mayor ganancia de suscriptores (+ el día siguiente)

In [ ]:
top3_dias_subs = evolucion.sort_values('subscribers_gained', ascending=False).head(3)
print('Los 3 días con más suscriptores ganados:')
display(top3_dias_subs[['fecha', 'dia_semana', 'subscribers_gained', 'views']])

In [ ]:
for _, fila in top3_dias_subs.iterrows():
    fecha = fila['fecha']
    videos_ese_dia = video[video['fecha_publicacion'] == fecha]

    dia_siguiente = evolucion[evolucion['fecha'] == fecha + pd.Timedelta(days=1)]
    subs_siguiente = dia_siguiente['subscribers_gained'].iloc[0] if len(dia_siguiente) else None

    print(f"\n{fecha.date()} ({fila['dia_semana']}): +{fila['subscribers_gained']} suscriptores")
    if len(videos_ese_dia):
        for _, v in videos_ese_dia.iterrows():
            print(f"  Publicado: [{v['formato']}] {v['categoria']} — '{v['titulo']}'")
    else:
        print('  (ningún vídeo publicado exactamente ese día)')
    if subs_siguiente is not None:
        variacion = subs_siguiente - fila['subscribers_gained']
        print(f'  Día siguiente: +{subs_siguiente} suscriptores ({variacion:+d} respecto al día pico)')
    else:
        print('  Día siguiente fuera del periodo analizado.')

## 6. ¿El calendario de publicación aprovecha el pico de audiencia (sábado)? — por formato

In [ ]:
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
esperado_sin_sesgo = 100 / 7

for formato in ['Short', 'Largo']:
    grupo = video[video['formato'] == formato]
    distribucion = grupo['dia_semana_publicacion'].value_counts(normalize=True).mul(100).reindex(orden_dias).fillna(0)
    pct_sabado = distribucion.get('Saturday', 0)
    diferencia_pp = esperado_sin_sesgo - pct_sabado

    print(f"\n--- {formato}s (n={len(grupo)}) ---")
    display(distribucion.round(1))
    if diferencia_pp > 3:
        print(f"Desalineación relevante: solo el {pct_sabado:.1f}% de los {formato.lower()}s se publica en sábado "
              f"({diferencia_pp:.1f} pp por debajo de lo esperado sin sesgo).")
    elif diferencia_pp < -3:
        print(f"Los {formato.lower()}s ya sobrerrepresentan el sábado ({pct_sabado:.1f}%).")
    else:
        print(f"Sin desalineación relevante para {formato.lower()}s ({pct_sabado:.1f}% vs. {esperado_sin_sesgo:.1f}% esperado).")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, formato, color in zip(axes, ['Short', 'Largo'], ['#B23A2E', '#2E6E9E']):
    grupo = video[video['formato'] == formato]
    dist = grupo['dia_semana_publicacion'].value_counts(normalize=True).mul(100).reindex(orden_dias).fillna(0)
    colores_barras = [color if d != 'Saturday' else '#F2A65A' for d in orden_dias]
    ax.bar(orden_dias, dist.values, color=colores_barras)
    ax.set_title(f'Calendario de publicación — {formato}s')
    ax.set_ylabel('% de vídeos')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 7. Tendencia reciente vs. histórico — separando Shorts de largos

Si el canal ha cambiado su mezcla de formatos recientemente, una comparación agregada 'reciente vs. histórico' confundiría ese cambio de mezcla con una mejora o empeoramiento real. Se separa por formato.

In [ ]:
for formato in ['Short', 'Largo']:
    grupo = video[video['formato'] == formato]
    recientes = grupo[grupo['es_reciente']]
    historicos = grupo[~grupo['es_reciente']]
    print(f"\n--- {formato}s: últimos 30 días (n={len(recientes)}) vs. histórico (n={len(historicos)}) ---")
    if len(recientes) == 0 or len(historicos) == 0:
        print('  Datos insuficientes en uno de los dos periodos para comparar.')
        continue
    print(f"  Vistas medias: {recientes['views_totales'].mean():.0f} vs. {historicos['views_totales'].mean():.0f}")
    print(f"  Retención media: {recientes['retencion_media'].mean()*100:.1f}% vs. {historicos['retencion_media'].mean()*100:.1f}%")
    variacion_views = (recientes['views_totales'].mean() / historicos['views_totales'].mean() - 1) * 100
    print(f"  Variación de vistas: {variacion_views:+.1f}%")

In [ ]:
# Tendencia de la MEZCLA de formatos a lo largo del tiempo: ¿el canal está produciendo más Shorts recientemente?
video_ordenado = video.sort_values('fecha_publicacion')
video_ordenado['bloque_50'] = np.arange(len(video_ordenado)) // 50
mezcla_por_bloque = video_ordenado.groupby('bloque_50')['es_short'].mean() * 100
print('% de Shorts por bloques de 50 vídeos consecutivos (orden cronológico):')
display(mezcla_por_bloque.round(1))

In [ ]:
# Tendencia de audiencia y suscriptores del canal (nivel canal, no se puede partir por formato:
# evolucion_diaria es un agregado del canal completo, no distingue qué formato generó las vistas del día)
dias_num = np.arange(len(evolucion))
pendiente_views, _ = np.polyfit(dias_num, evolucion['views'], 1)
pendiente_ingresos, _ = np.polyfit(dias_num, ingresos.sort_values('fecha')['estimated_revenue'], 1)
print(f"Tendencia de vistas diarias del canal: {pendiente_views:+.1f} vistas/día")
print(f"Tendencia de ingreso diario del canal: {pendiente_ingresos:+.3f} EUR/día")
direccion = 'MEJORANDO' if pendiente_views > 0 and pendiente_ingresos > 0 else ('EMPEORANDO' if pendiente_views < 0 and pendiente_ingresos < 0 else 'MIXTA')
print(f"Dirección general del periodo: {direccion}")

## Conclusiones y recomendaciones accionables

*(Completar tras revisar los números reales de tu ejecución.)*

1. **Días fuertes:** si predominan Shorts o largos entre lo publicado esos días, indica qué formato empuja realmente el pico — no asumir que es 'más contenido' en general.
2. **Categoría Afición:** comparar siempre Shorts-con-Shorts y largos-con-largos frente al resto del canal; decidir potenciarla solo si supera a su propio equivalente de formato, no al canal mezclado.
3. **Vídeos virales:** con el z-score corregido por formato, un viral lo es de verdad dentro de su categoría y su formato — más fiable que compararlo contra todo el catálogo mezclado.
4. **Vídeos "perfectos":** dos Top 5 independientes (Shorts y largos) son más útiles que uno mezclado: cada uno sirve de referencia para su propia línea de producción.
5. **Ganancia de suscriptores:** el formato publicado en los días de pico de suscriptores es una pista de qué tipo de contenido convierte espectador en suscriptor, no solo en vista.
6. **Calendario vs. audiencia real:** si Shorts y largos tienen calendarios de publicación distintos, la recomendación de "publicar más en sábado" puede aplicar a un formato y no al otro — no tratarlo como una única política de canal.
7. **Tendencia y mezcla de formatos:** si el canal está produciendo más Shorts recientemente Y el ingreso cae, vale la pena revisar si el cambio de mezcla de formato está detrás de la caída antes de recomendar más Shorts como solución genérica.